# 03 · Final model, interpretation, live demo and conclusions
**UE24CS352A – Machine Learning · Mini-Project (Problem #101)** · **Team:** *<Name 1 – SRN>*, *<Name 2 – SRN>*

**Owner:** *<Name 2>* (demo) with *<Name 1>* reviewing conclusions · **Reads:** outputs of notebooks 01 and 02 · **Writes:** `reports/figures/06_ridge_coefficients.png`

Demo helpers live in `src/demo.py`.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))          # lets us import the shared code in src/
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_PROC, FIG_DIR, SEED, PRIMARY_MODEL, ensure_dirs
from src.features import D_COLS
from src.models import MODELS
from src.demo import predict_matchup, pickem_sheet
try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50, "display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
np.random.seed(SEED)
ensure_dirs()
print("Project root:", ROOT)

In [ ]:
bowls = pd.read_csv(DATA_PROC / "bowl_matchups.csv", parse_dates=["date"])
print("Bowl games:", len(bowls))

## 2. Final model and interpretation
We train the primary model on **all** bowl games and inspect its (standardised) coefficients.

In [ ]:
final = MODELS[PRIMARY_MODEL]().fit(bowls)
coef = pd.Series(final.est_[-1].coef_, index=[c[2:] for c in final.cols_]).sort_values(key=np.abs, ascending=False)
print("Selected ridge alpha:", round(float(final.est_[-1].alpha_), 2))
top = coef.head(12)[::-1]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.barh(top.index, top.values, color=np.where(top.values > 0, "#54A24B", "#E45756"))
ax.set_xlabel("Standardised coefficient (effect on predicted margin)")
ax.set_title("Top drivers — Differential Ridge (all data)")
plt.tight_layout(); plt.savefig(FIG_DIR / "06_ridge_coefficients.png"); plt.show()
display(coef.round(3).to_frame("coef").T)

## 3. Live demo
`predict_matchup` predicts a single game. `pickem_sheet` produces a full Pick'em sheet for a season. When the season is in
our data, **that season is excluded from training**, so the demo does not cheat.

In [ ]:
# ---- DEMO: edit the season / teams below ----
demo_season = 2010
g0 = bowls[bowls.season == demo_season].iloc[-1]          # the last bowl game of that season
predict_matchup(g0.team_a, g0.team_b, demo_season)
print("Actual result:", g0.team_a if g0.margin > 0 else g0.team_b, "won by", int(abs(g0.margin)))

In [ ]:
display(pickem_sheet(demo_season).head(15))

## 10. Conclusions
*(Update the numbers below from the results table when you finalise your write-up.)*

**What we did.** We built a leakage-free pipeline on the 2004–2010 box-score data: team-season features from regular-season games only,
223 bowl games, symmetric models on feature differences, and 7-fold leave-one-season-out CV with all tuning and feature selection inside the training folds.

**Results (pooled CV accuracy, 223 games).**
- Best models reached roughly **57–60 %**: SVM (RBF) 60.1 %, Score Regression 59.2 %, Differential OLS + p-value selection / Logistic / Random Forest about 57 %.
- Our pre-specified primary model, **Differential Ridge, scored 55.6 %**, which is above the coin / listed-home-team baselines (about 53 %) but not significantly (p = 0.11 vs 50 %).
- A one-line rule, *"pick the team with the higher average scoring margin"*, scored 58.7 %, which **matches the more complex models**. Regular-season win % alone was no better than chance (52 %).

**Confidence works.** For the primary model, accuracy rises with |predicted margin|: about **45 % → 58 % → 64 %** across low / medium / high confidence terciles.
This is the property Pick'em needs. In the simulated Pick'em game the models earned about 60 % of the maximum points, against 50 % for a random picker.

**Honest comparison with the reference paper (66 %).** We did **not** reproduce 66 %. Plausible reasons:
- we have fewer games (223 vs ~350) and different, fewer features (no returns, field goals or touchdowns);
- our protocol is stricter, since selection and tuning never see the held-out season, whereas the paper's p-value feature selection used regressions across all 11 seasons, which is likely optimistic;
- with ~30 games per season, one season's accuracy has a standard error of about ±9 points.

**Statistical caution.** The 95 % confidence intervals are about ±6 points wide, so most model differences are **not statistically significant**.
Treat the ranking of the top models as indicative, not definitive.

**Limitations.** Small sample; bowl games are driven by motivation, opt-outs and injuries that box scores cannot capture; `fum` is fumbles and not fumbles lost;
no rankings or recruiting data; FCS opponents add noise to strength of schedule.

**Future work.** (1) add AP rank / conference / Elo-style ratings, (2) PCA or other dimensionality reduction, (3) repeated or nested CV with paired tests (McNemar) to compare models,
(4) an upset-detection model, (5) extend the data to later seasons for more training games.